In [2]:
!python --version



Python 3.10.18


In [3]:
!python -m pip install --upgrade pip setuptools wheel


  Using cached pip-25.3-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-25.3-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 25.2
    Uninstalling pip-25.2:
      Successfully uninstalled pip-25.2


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [5]:
!pip install numpy opencv-python



In [6]:
!pip install tensorflow==2.12.1


ERROR: Could not find a version that satisfies the requirement tensorflow==2.12.1 (from versions: 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0)
ERROR: No matching distribution found for tensorflow==2.12.1


In [ ]:
import cv2
import os
import numpy as np
import mediapipe as mp
from deepface import DeepFace
from collections import deque

# ======================
# CONFIG
# ======================
BASE_DIR = "mimik"
EMOTIONS = ["happy", "sad", "neutral"]
MAX_REF = 30
THRESHOLD = 0.10  # %10

os.makedirs(BASE_DIR, exist_ok=True)

# ======================
# MEDIAPIPE
# ======================
mp_face = mp.solutions.face_mesh
face_mesh = mp_face.FaceMesh(static_image_mode=False)

# ======================
# UTILS
# ======================
def get_sol_yanak(lm, w, h):
    # Ağız sol köşe (61) – sol göz altı (234)
    p1 = lm.landmark[61]
    p2 = lm.landmark[234]

    x1, y1 = int(p1.x * w), int(p1.y * h)
    x2, y2 = int(p2.x * w), int(p2.y * h)

    return np.linalg.norm(np.array([x1, y1]) - np.array([x2, y2]))

def load_refs(emotion, region):
    path = f"{BASE_DIR}/{emotion}/{region}.npy"
    if not os.path.exists(path):
        return deque(maxlen=MAX_REF)
    return deque(np.load(path).tolist(), maxlen=MAX_REF)

def save_refs(emotion, region, data):
    os.makedirs(f"{BASE_DIR}/{emotion}", exist_ok=True)
    np.save(f"{BASE_DIR}/{emotion}/{region}.npy", np.array(data))

# ======================
# MAIN
# ======================
cap = cv2.VideoCapture(0)

print("ESC ile çık")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    h, w, _ = frame.shape
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # 1️⃣ DUYGU ANALİZİ
    try:
        emo_res = DeepFace.analyze(
            rgb,
            actions=["emotion"],
            enforce_detection=False
        )[0]["emotion"]
    except:
        continue

    # normalize (0–1)
    emo_scores = {k: v / 100.0 for k, v in emo_res.items()}

    # 2️⃣ YÜZ LANDMARK
    res = face_mesh.process(rgb)
    if not res.multi_face_landmarks:
        continue

    lm = res.multi_face_landmarks[0]
    sol_yanak = get_sol_yanak(lm, w, h)

    print("\nSol yanak ölçüm:", round(sol_yanak, 2))

    # 3️⃣ HER DUYGU İÇİN AYRI AYRI KIYAS
    diffs = []

    for emo in EMOTIONS:
        refs = load_refs(emo, "sol_yanak")

        if len(refs) >= 5:
            mean_ref = np.mean(refs)
            diff = (sol_yanak - mean_ref) / mean_ref
            diffs.append(diff)

            print(f"{emo}: %{round(diff*100,1)} fark")
        else:
            print(f"{emo}: referans oluşturuluyor")

        # 4️⃣ REFERANS GÜNCELLE
        refs.append(sol_yanak)
        save_refs(emo, "sol_yanak", refs)

    # 5️⃣ ORTALAMA KARAR
    if diffs:
        avg = np.mean(diffs)
        if abs(avg) > THRESHOLD:
            print("⚠️ Genel yanak hareketi belirgin farklılıkta")
        else:
            print("✅ Genel yanak hareketi normal")

    cv2.imshow("cam", frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


: 